# Module 11 - 실습 3 솔루션: 골든셋 구성 + 자동 평가

In [ ]:
import sys
sys.path.insert(0, '../..')

import json
import re
from common.fake_llm import FakeLLM

print("골든셋 평가 솔루션 준비 완료")

## 실습 1 솔루션: Golden Set 정의

In [ ]:
GOLDEN_SET = [
    {
        "case_id": "case_001",
        "description": "NullPointerException 분석",
        "input": {
            "query": "getUser() 메서드에서 NullPointerException이 발생합니다."
        },
        "expected_output": {
            "summary": "null 체크 누락",
            "confidence": 0.85,
            "issues": ["null 체크 누락"],
            "recommendation": "null 체크 추가"
        },
        "evaluation_criteria": [
            {"name": "has_summary", "check_type": "field_exists", "field": "summary", "weight": 1.0},
            {"name": "confidence_ok", "check_type": "value_range", "field": "confidence",
             "min_value": 0.5, "max_value": 1.0, "weight": 0.8},
            {"name": "has_issues", "check_type": "list_min_length", "field": "issues",
             "min_length": 1, "weight": 0.5},
            {"name": "has_recommendation", "check_type": "field_exists", "field": "recommendation", "weight": 0.7},
        ]
    },
    {
        "case_id": "case_002",
        "description": "보안 취약점 분석",
        "input": {
            "query": "로그인 API에서 SQL injection 취약점이 발견되었습니다."
        },
        "expected_output": {
            "summary": "SQL injection 취약점",
            "confidence": 0.9,
            "issues": ["파라미터 검증 미흡"],
            "recommendation": "Prepared Statement 사용"
        },
        "evaluation_criteria": [
            {"name": "has_summary", "check_type": "field_exists", "field": "summary", "weight": 1.0},
            {"name": "confidence_high", "check_type": "value_range", "field": "confidence",
             "min_value": 0.6, "max_value": 1.0, "weight": 0.8},
            {"name": "has_issues", "check_type": "list_min_length", "field": "issues",
             "min_length": 1, "weight": 0.5},
        ]
    },
    {
        "case_id": "case_003",
        "description": "성능 문제 분석",
        "input": {
            "query": "데이터베이스 쿼리가 N+1 문제를 일으키고 있습니다."
        },
        "expected_output": {
            "summary": "N+1 쿼리 문제",
            "confidence": 0.8,
            "issues": ["불필요한 반복 쿼리"],
            "recommendation": "eager loading 적용"
        },
        "evaluation_criteria": [
            {"name": "has_summary", "check_type": "field_exists", "field": "summary", "weight": 1.0},
            {"name": "confidence_ok", "check_type": "value_range", "field": "confidence",
             "min_value": 0.5, "max_value": 1.0, "weight": 0.8},
            {"name": "has_recommendation", "check_type": "field_exists", "field": "recommendation", "weight": 0.7},
        ]
    },
]

In [ ]:
assert len(GOLDEN_SET) >= 1
for case in GOLDEN_SET:
    assert "case_id" in case
    assert "input" in case
    assert "evaluation_criteria" in case
    assert len(case["evaluation_criteria"]) >= 1
print(f"✅ 실습 1 완료! {len(GOLDEN_SET)}개 케이스의 Golden Set이 준비되었습니다.")

## 실습 2 솔루션: _run_check 함수

In [ ]:
def _run_check(criterion: dict, actual_output: dict) -> bool:
    """개별 평가 기준을 실행합니다."""
    check_type = criterion["check_type"]
    field = criterion["field"]
    value = actual_output.get(field)
    
    if check_type == "field_exists":
        return value is not None and value != ""
    
    elif check_type == "value_range":
        if value is None:
            return False
        min_val = criterion.get("min_value", float("-inf"))
        max_val = criterion.get("max_value", float("inf"))
        return min_val <= float(value) <= max_val
    
    elif check_type == "list_min_length":
        if not isinstance(value, list):
            return False
        return len(value) >= criterion.get("min_length", 1)
    
    elif check_type == "exact_match":
        return value == criterion.get("expected_value")
    
    elif check_type == "contains":
        return criterion.get("substring", "") in str(value)
    
    return False

In [ ]:
test_output = {"summary": "분석 완료", "confidence": 0.85, "issues": ["이슈1"], "recommendation": "조치"}

assert _run_check({"check_type": "field_exists", "field": "summary"}, test_output) == True
assert _run_check({"check_type": "field_exists", "field": "missing_field"}, test_output) == False
assert _run_check({"check_type": "value_range", "field": "confidence", "min_value": 0.5, "max_value": 1.0}, test_output) == True
assert _run_check({"check_type": "value_range", "field": "confidence", "min_value": 0.9, "max_value": 1.0}, test_output) == False
assert _run_check({"check_type": "list_min_length", "field": "issues", "min_length": 1}, test_output) == True
assert _run_check({"check_type": "list_min_length", "field": "issues", "min_length": 5}, test_output) == False
print("✅ 실습 2 완료! _run_check가 모든 check_type을 처리합니다.")

## 실습 3 솔루션: evaluate_single_case

In [ ]:
def evaluate_single_case(case: dict, actual_output: dict) -> dict:
    """단일 Golden Case를 평가합니다."""
    criteria = case.get("evaluation_criteria", [])
    checks = []
    total_weight = 0.0
    weighted_score = 0.0
    
    for criterion in criteria:
        weight = criterion.get("weight", 1.0)
        total_weight += weight
        
        passed = _run_check(criterion, actual_output)
        checks.append({
            "name": criterion["name"],
            "passed": passed,
            "weight": weight,
        })
        
        if passed:
            weighted_score += weight
    
    score = weighted_score / total_weight if total_weight > 0 else 0.0
    
    return {
        "case_id": case["case_id"],
        "score": round(score, 3),
        "passed": score >= 0.8,
        "checks": checks,
    }

In [ ]:
good_output = {"summary": "null 체크 누락", "confidence": 0.85, "issues": ["null 체크"], "recommendation": "조치"}
eval_result = evaluate_single_case(GOLDEN_SET[0], good_output)

assert "case_id" in eval_result
assert "score" in eval_result
assert "passed" in eval_result
assert 0.0 <= eval_result["score"] <= 1.0
assert eval_result["passed"] == True

print(f"평가 결과: 점수={eval_result['score']:.3f}, 통과={eval_result['passed']}")
print("✅ 실습 3 완료! evaluate_single_case가 올바르게 평가합니다.")

## 실습 4 솔루션: 전체 Golden Set 평가

In [ ]:
fake_llm = FakeLLM(responses={
    "NullPointerException": '{"summary": "null 체크 누락", "confidence": 0.85, "issues": ["null 체크 누락"], "recommendation": "조건문 추가"}',
    "default": '{"summary": "일반적인 분석 결과", "confidence": 0.75, "issues": ["확인 필요"], "recommendation": "코드 검토"}',
})

def simple_parse(text):
    try:
        return json.loads(text)
    except:
        m = re.search(r'\{.*\}', text, re.DOTALL)
        if m:
            return json.loads(m.group())
        return {}

def llm_function(input_data: dict) -> dict:
    query = input_data.get("query", "")
    response = fake_llm.invoke(query).content
    return simple_parse(response)

# 전체 평가 실행
all_results = []
for case in GOLDEN_SET:
    actual = llm_function(case["input"])
    result = evaluate_single_case(case, actual)
    all_results.append(result)
    print(f"  [{case['case_id']}] 점수={result['score']:.3f}, 통과={result['passed']}")

avg_score = sum(r["score"] for r in all_results) / len(all_results)
passed_count = sum(1 for r in all_results if r["passed"])
print(f"\n평균 점수: {avg_score:.3f}")
print(f"통과: {passed_count}/{len(all_results)}")

In [ ]:
assert len(all_results) == len(GOLDEN_SET)
for r in all_results:
    assert "score" in r and "passed" in r
print(f"✅ 실습 4 완료! {len(GOLDEN_SET)}개 케이스 평가 완료.")